In [1]:
from ultralytics import YOLO

model = YOLO("yolov8s-worldv2.pt")

In [2]:
print(type(model))
print(model.ckpt_path)

<class 'ultralytics.models.yolo.model.YOLOWorld'>
yolov8s-worldv2.pt


In [7]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [13]:
classes = [
    "truck",
    "excavator",
    "concrete mixer"
]

In [11]:
results = model.predict(
    source=r"C:\Users\Ева\Desktop\Хакатон\data_preproc\constraction_labelme\images\0a12ef44-ee64-4b68-90cf-92edb47dbe16_jpg.rf.65bb71f61613aedf7db83d2a99dc09bf.jpg",
    conf=0.1,
    device=0
)


image 1/1 C:\Users\\Desktop\\data_preproc\constraction_labelme\images\0a12ef44-ee64-4b68-90cf-92edb47dbe16_jpg.rf.65bb71f61613aedf7db83d2a99dc09bf.jpg: 384x640 1 truck, 41.9ms
Speed: 2.2ms preprocess, 41.9ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)


In [12]:
results[0].show()

Прогон на датасете

In [26]:
from ultralytics import YOLO
from pathlib import Path

# Корень датасета
dataset_path = Path(r"C:\Users\Ева\Desktop\Хакатон\data_preproc\constraction_labelme")

# Папки
images_path = dataset_path / "images"
labels_path = dataset_path / "labels"

# Папка для результатов
results_path = Path("yoloworld_results")
results_images_path = results_path / "images"

results_images_path.mkdir(parents=True, exist_ok=True)

# Модель
model = YOLO("yolov8s-worldv2.pt")

# Наши классы
classes = [
    "a road transport truck",
    "a construction excavator",
    "a concrete mixer truck"
]

model.set_classes(classes)

print("Модель загружена")
print("Классы:", classes)
print("Результаты будут сохраняться в:", results_path)

c:\Users\Ева\Desktop\Хакатон\venv\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Модель загружена
Классы: ['a road transport truck', 'a construction excavator', 'a concrete mixer truck']
Результаты будут сохраняться в: yoloworld_results


In [29]:
from pathlib import Path

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

image_files = [
    p for p in images_path.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

print(f"Найдено изображений: {len(image_files)}")

Найдено изображений: 3651


Запуск прогона

In [28]:
import json
from tqdm.auto import tqdm

# Минимальная уверенность для отображения детекций
conf_threshold = 0.1

# Все предсказания будем дополнительно сохранять в JSON
all_predictions = {}

for image_file in tqdm(image_files, desc="Обработка изображений"):
    
    # Предсказание
    results = model.predict(
        source=str(image_file),
        conf=conf_threshold,
        verbose=False
    )
    
    result = results[0]
    
    # Сохраняем информацию о предсказаниях
    image_predictions = []
    
    if result.boxes is not None:
        boxes = result.boxes
        
        for i in range(len(boxes)):
            # Координаты бокса
            xyxy = boxes.xyxy[i].cpu().numpy().tolist()
            
            # Confidence
            confidence = float(boxes.conf[i].cpu().item())
            
            # Индекс класса
            class_id = int(boxes.cls[i].cpu().item())
            
            # Название класса
            class_name = classes[class_id]
            
            image_predictions.append({
                "class": class_name,
                "confidence": confidence,
                "bbox": xyxy
            })
    
    all_predictions[image_file.name] = image_predictions
    
    # Рисуем результат
    annotated_image = result.plot(
        line_width=2,
        font_size=12,
        labels=True,
        boxes=True,
        conf=True
    )
    
    # Сохраняем изображение
    output_file = results_images_path / image_file.name
    
    import cv2
    cv2.imwrite(
        str(output_file),
        annotated_image
    )

print("\nГотово!")
print(f"Результаты сохранены в: {results_images_path}")

Обработка изображений: 100%|██████████| 3651/3651 [01:25<00:00, 42.49it/s]


Готово!
Результаты сохранены в: yoloworld_results\images


In [30]:
predictions_file = results_path / "predictions.json"

with open(predictions_file, "w", encoding="utf-8") as f:
    json.dump(
        all_predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Предсказания сохранены: {predictions_file}")

Предсказания сохранены: yoloworld_results\predictions.json


In [31]:
import json

# Берём первый JSON
sample_json = next(labels_path.glob("*.json"))

with open(sample_json, "r", encoding="utf-8") as f:
    sample = json.load(f)

print("Файл:", sample_json.name)
print("Количество объектов:", len(sample.get("shapes", [])))

for shape in sample.get("shapes", [])[:5]:
    print()
    print("label:", shape.get("label"))
    print("shape_type:", shape.get("shape_type"))
    print("points:", shape.get("points"))

Файл: 000001_jpg.rf.758184e127cbc907e8c8cc8dd9974125.json
Количество объектов: 2

label: экскаватор
shape_type: polygon
points: [[173.0, 19.0], [398.0, 19.0], [398.0, 302.0], [173.0, 302.0]]

label: грузовик
shape_type: polygon
points: [[31.0, 168.0], [233.0, 168.0], [233.0, 289.0], [31.0, 289.0]]


Перевод polygon в Bbox

In [32]:
import json
from pathlib import Path

# Соответствие LabelMe → классы YOLO-World
label_to_class = {
    "грузовик": "truck",
    "экскаватор": "excavator",
    "бетономешалка": "concrete mixer"
}


def polygon_to_bbox(points):
    """
    Переводит polygon LabelMe в bbox:
    [xmin, ymin, xmax, ymax]
    """
    xs = [point[0] for point in points]
    ys = [point[1] for point in points]

    return [
        min(xs),
        min(ys),
        max(xs),
        max(ys)
    ]


# Читаем всю LabelMe-разметку
ground_truth = {}

json_files = list(labels_path.glob("*.json"))

for json_file in json_files:

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_name = json_file.stem

    gt_objects = []

    for shape in data.get("shapes", []):

        label = shape.get("label")

        # Пропускаем неизвестные классы
        if label not in label_to_class:
            continue

        points = shape.get("points", [])

        if len(points) < 2:
            continue

        bbox = polygon_to_bbox(points)

        gt_objects.append({
            "class": label_to_class[label],
            "original_class": label,
            "bbox": bbox
        })

    ground_truth[image_name] = gt_objects


print("Обработано файлов:", len(ground_truth))

# Статистика
from collections import Counter

gt_counts = Counter()

for objects in ground_truth.values():
    for obj in objects:
        gt_counts[obj["class"]] += 1

print("\nGround Truth:")
for class_name, count in gt_counts.items():
    print(f"{class_name:20} {count}")

Обработано файлов: 3651

Ground Truth:
excavator            2359
truck                4057
concrete mixer       1118


In [34]:
print("Пример GT:")
for name, objects in list(ground_truth.items())[:1]:
    print(name)
    print(objects)

print("\nПример predictions:")
for name, objects in list(all_predictions.items())[:1]:
    print(Path(name).stem)
    print(objects)

Пример GT:
000001_jpg.rf.758184e127cbc907e8c8cc8dd9974125
[{'class': 'excavator', 'original_class': 'экскаватор', 'bbox': [173.0, 19.0, 398.0, 302.0]}, {'class': 'truck', 'original_class': 'грузовик', 'bbox': [31.0, 168.0, 233.0, 289.0]}]

Пример predictions:
000001_jpg.rf.758184e127cbc907e8c8cc8dd9974125
[{'class': 'a concrete mixer truck', 'confidence': 0.725829541683197, 'bbox': [29.518068313598633, 170.9505157470703, 235.115478515625, 291.74005126953125]}, {'class': 'a road transport truck', 'confidence': 0.5178797245025635, 'bbox': [29.565279006958008, 170.61741638183594, 235.2212371826172, 291.7184753417969]}, {'class': 'a road transport truck', 'confidence': 0.41659778356552124, 'bbox': [4.253762722015381, 15.851644515991211, 402.448974609375, 304.3528747558594]}, {'class': 'a road transport truck', 'confidence': 0.32965636253356934, 'bbox': [165.28709411621094, 19.008651733398438, 400.7845764160156, 304.15740966796875]}, {'class': 'a road transport truck', 'confidence': 0.20628

Метрики

In [35]:
import numpy as np
from collections import defaultdict


def calculate_iou(box1, box2):
    """
    IoU для bbox формата:
    [xmin, ymin, xmax, ymax]
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)

    intersection = intersection_width * intersection_height

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - intersection

    if union == 0:
        return 0.0

    return intersection / union


# Порог IoU
iou_threshold = 0.5

# Статистика по классам
stats = {
    class_name: {
        "TP": 0,
        "FP": 0,
        "FN": 0
    }
    for class_name in classes
}


# Дополнительно сохраним информацию о каждом prediction
match_results = {}


# Перебираем изображения
for image_stem, gt_objects in ground_truth.items():

    predictions = all_predictions.get(
        image_stem + ".jpg",
        []
    )

    # Если имя файла имеет другое расширение,
    # попробуем найти через stem
    if not predictions:

        for filename, preds in all_predictions.items():
            if Path(filename).stem == image_stem:
                predictions = preds
                break

    # Какие GT уже были сопоставлены
    matched_gt = set()

    image_matches = []

    # Сначала сортируем prediction
    # от большей уверенности к меньшей
    predictions_sorted = sorted(
        predictions,
        key=lambda x: x["confidence"],
        reverse=True
    )

    for pred in predictions_sorted:

        pred_class = pred["class"]
        pred_bbox = pred["bbox"]
        confidence = pred["confidence"]

        best_iou = 0
        best_gt_index = None

        # Ищем лучший GT того же класса
        for gt_index, gt in enumerate(gt_objects):

            if gt_index in matched_gt:
                continue

            if gt["class"] != pred_class:
                continue

            iou = calculate_iou(
                pred_bbox,
                gt["bbox"]
            )

            if iou > best_iou:
                best_iou = iou
                best_gt_index = gt_index

        # Нашли совпадение
        if best_gt_index is not None and best_iou >= iou_threshold:

            stats[pred_class]["TP"] += 1
            matched_gt.add(best_gt_index)

            image_matches.append({
                "prediction": pred,
                "type": "TP",
                "iou": best_iou
            })

        else:

            stats[pred_class]["FP"] += 1

            image_matches.append({
                "prediction": pred,
                "type": "FP",
                "iou": best_iou
            })

    # Все GT, которые не были найдены
    for gt_index, gt in enumerate(gt_objects):

        if gt_index not in matched_gt:

            stats[gt["class"]]["FN"] += 1

            image_matches.append({
                "ground_truth": gt,
                "type": "FN"
            })

    match_results[image_stem] = image_matches


# Вывод
print("Результаты при IoU =", iou_threshold)
print()

print(
    f"{'Класс':20}"
    f"{'TP':>8}"
    f"{'FP':>8}"
    f"{'FN':>8}"
)

print("-" * 44)

for class_name in classes:

    s = stats[class_name]

    print(
        f"{class_name:20}"
        f"{s['TP']:8}"
        f"{s['FP']:8}"
        f"{s['FN']:8}"
    )

KeyError: 'excavator'